# 📚 livros-converter — Colab Free (T4 GPU)

Pipeline OCR + LLM para converter livros didáticos em Markdown estruturado para RAG.

**Antes de começar:**
1. Ative GPU: `Editar → Configurações do notebook → T4 GPU`
2. Tenha sua **chave Gemini** pronta: https://aistudio.google.com/apikey
3. Tenha o(s) PDF(s) em `MyDrive/livros-input/`

**Tempo estimado para 1 capítulo (~26 págs):** ~12 min

## 1. Verificar GPU

In [ ]:
!nvidia-smi | head -20
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(),
      'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/livros-input', exist_ok=True)
os.makedirs('/content/drive/MyDrive/livros-output', exist_ok=True)
os.makedirs('/content/drive/MyDrive/hf_cache', exist_ok=True)

# Linka cache HF pro Drive (evita rebaixar GLM-OCR a cada sessão)
!mkdir -p /root/.cache && rm -rf /root/.cache/huggingface
!ln -sf /content/drive/MyDrive/hf_cache /root/.cache/huggingface

print('OK')

## 3. Clonar repositório

Se o repo é **privado**, vai pedir login. Use seu PAT (Personal Access Token) do GitHub como senha.

In [ ]:
%cd /content
!rm -rf livros-converter
!git clone https://github.com/Goncalui/livros-converter.git
%cd livros-converter
!ls

## 4. Instalar dependências (Node + Python)

~5 min na primeira vez (baixa torch + paddle + transformers).

In [ ]:
%%bash
set -e

# Node 22 (Colab vem com versão mais antiga)
if ! node -v 2>/dev/null | grep -q v22; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1
  apt-get install -y nodejs >/dev/null 2>&1
fi
node --version

# Deps Node
npm install --silent

# Deps Python
pip install -q PyMuPDF Pillow pytesseract \
  paddlepaddle==3.0.0 paddleocr==3.5.0 \
  glmocr 'transformers>=5.0' accelerate sentencepiece pypdfium2 opencv-python-headless

# Tesseract com pt
apt-get install -y -q tesseract-ocr tesseract-ocr-por
echo 'tesseract:'; tesseract --version 2>&1 | head -1
echo 'OK — deps instaladas'

## 5. Configurar `.env` com chave Gemini

**Substitua** `COLE_AQUI_SUA_CHAVE` pela sua chave real.

In [ ]:
GEMINI_API_KEY = 'COLE_AQUI_SUA_CHAVE_AIza...'  # ← edite aqui

env = f'''# OCR
OCR_MODE=compare
PADDLE_OCR_LANG=pt
PADDLE_OCR_VERSION=PP-OCRv5
PADDLE_FORCE_MOBILE=1
TESSERACT_CMD=/usr/bin/tesseract
TESSERACT_LANG=por
COMPARE_INCLUDE_TESSERACT=1
OCR_MIN_CONFIDENCE=0.80

# GLM-OCR (transformers direct, GPU)
GLM_OCR_MODE=selfhosted
CUDA_VISIBLE_DEVICES=0
GLM_LAYOUT_DEVICE=cuda
GLM_OCR_MAX_NEW_TOKENS=4096

# LLM — Gemini como primário (Claude CLI não roda no Colab)
LLM_ORDER_TEXT=gemini,ollama
LLM_ORDER_VISION=gemini,ollama
GEMINI_API_KEY={GEMINI_API_KEY}
GEMINI_TEXT_MODEL=gemini-2.5-pro
GEMINI_VISION_MODEL=gemini-2.5-pro

# Pipeline
BATCH_SIZE=5
BATCH_OVERLAP=1
PYTHON_BIN=python3
'''

with open('/content/livros-converter/.env', 'w') as f:
    f.write(env)

assert 'COLE_AQUI' not in GEMINI_API_KEY, '⚠️ Substitua a chave Gemini antes de continuar!'
print('.env criado. Chave Gemini: ' + GEMINI_API_KEY[:10] + '…')

## 6. Recortar capítulo do PDF

Edite `PDF`, `PAGE_FROM` e `PAGE_TO` conforme seu livro.
Rode primeiro com o sumário aberto (último print) pra escolher o intervalo.

In [ ]:
import fitz, os

PDF        = '/content/drive/MyDrive/livros-input/Biologia - Volume 1.pdf'
OUT_DIR    = '/content/livros-converter/input-bio-cap1'
PAGE_FROM  = 15   # 1-indexado
PAGE_TO    = 40   # inclusive

doc = fitz.open(PDF)
print(f'PDF tem {len(doc)} páginas')
print('--- sumário ---')
for level, title, page in doc.get_toc():
    print(f'L{level} pág{page:>4}  {"  "*level}{title}')
print()

os.makedirs(OUT_DIR, exist_ok=True)
for i in range(PAGE_FROM - 1, PAGE_TO):
    pix = doc[i].get_pixmap(dpi=200)
    pix.save(f'{OUT_DIR}/page-{i - PAGE_FROM + 2:03d}.png')
print(f'OK — {PAGE_TO - PAGE_FROM + 1} páginas extraídas em {OUT_DIR}')

## 7. (Opcional) UI web via ngrok

Pegue um auth token grátis em https://dashboard.ngrok.com/get-started/your-authtoken e cole abaixo.
Pule esta célula se não quiser UI.

In [ ]:
NGROK_TOKEN = ''  # ← cole seu token (deixe vazio pra pular)

if NGROK_TOKEN:
    !pip install -q pyngrok
    from pyngrok import ngrok
    import subprocess, time
    ngrok.set_auth_token(NGROK_TOKEN)
    subprocess.Popen(['node', 'src/cli.js', 'ui', '5173'], cwd='/content/livros-converter',
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    public = ngrok.connect(5173)
    print('🌐 UI:', public.public_url)
else:
    print('UI desativada (sem token).')

## 8. Rodar pipeline

Saída em tempo real. Estimativa T4: OCR ~8min · Convert Gemini ~3min · Total ~12min.

Se travar/desconectar, rode `node src/cli.js resume input-bio-cap1` numa nova célula — retoma do checkpoint.

In [ ]:
!cd /content/livros-converter && PYTHONIOENCODING=utf-8 node src/cli.js convert ./input-bio-cap1 2>&1 | tee workspace/run.log

## 9. Salvar resultado no Drive

In [ ]:
%%bash
SLUG=input-bio-cap1
SRC=/content/livros-converter/workspace/$SLUG/output
DST=/content/drive/MyDrive/livros-output/$SLUG
mkdir -p "$DST"
cp -rv "$SRC"/* "$DST"/
echo '---'
ls -la "$DST"

## 10. (Opcional) Mostrar markdown gerado

In [ ]:
from IPython.display import Markdown, display
import glob, os

out_dir = '/content/livros-converter/workspace/input-bio-cap1/output'
files = sorted(glob.glob(f'{out_dir}/cap-*.md'))
print(f'{len(files)} capítulos gerados:')
for f in files:
    print(' -', os.path.basename(f), f'({os.path.getsize(f)//1024} KB)')

if files:
    print('\n=== Preview do primeiro capítulo (3000 chars) ===')
    with open(files[0], 'r', encoding='utf-8') as f:
        content = f.read()
    display(Markdown(content[:3000] + '\n\n…'))

## 11. (Opcional) Comparação OCR — auditar Paddle vs GLM vs Tesseract

Mostra qual engine ganhou em cada página e métricas brutas.

In [ ]:
import json, glob
from collections import Counter

raw = '/content/livros-converter/workspace/input-bio-cap1/raw'
files = sorted(glob.glob(f'{raw}/page-*.compare.json'))
winners = Counter()
rows = []
for f in files:
    j = json.load(open(f))
    winners[j.get('winner') or 'none'] += 1
    eng = j.get('engines', {})
    rows.append({
        'pág': j['page'],
        'vencedor': j.get('winner'),
        **{f'{k}_score': eng.get(k, {}).get('score') for k in ['paddle','glm','tesseract']}
    })

print('=== Vencedores ===')
for k, v in winners.most_common():
    print(f'  {k}: {v}')

try:
    import pandas as pd
    print('\n=== Tabela ===')
    print(pd.DataFrame(rows).to_string(index=False))
except ImportError:
    for r in rows: print(r)

---

## 🎯 Pronto!

Resultado salvo em `MyDrive/livros-output/input-bio-cap1/`:
- `index.md` — sumário do capítulo
- `cap-NN-*.md` — markdown estruturado por capítulo
- `_full.md` — versão concatenada
- `_validation.json` — avisos de validação (YAML, IDs, fórmulas)

**Para rodar outro capítulo:** volte na Célula 6, ajuste `PAGE_FROM`/`PAGE_TO` e `OUT_DIR`, depois rode 6 → 8 → 9.